## 1. 加载环境变量

In [52]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")

## 2. 定义工具

In [53]:
from langchain_core.tools import tool
from datetime import datetime

@tool
def get_weather(location: str) -> str:
    """获取指定城市的天气情况"""
    return f"{location}今天天气晴朗，气温25度。."

@tool
def get_current_time() -> str:
    """获取当前地区时间"""
    now = datetime.now()
    return f"当前时间: {now.strftime('%Y-%m-%d %H:%M:%S')}"

tools = [get_weather, get_current_time]

## 3. 创建 Agent

In [54]:
from langchain_community.chat_models import ChatTongyi
from langchain.agents import create_agent

# 1. 初始化通义千问大模型，并设置参数
llm = ChatTongyi(
    model="qwen-plus",
    temperature=2,      # 设置模型温度 (0.0-2.0)，值越大越有创造性，值越小越严谨
    streaming=False       # 暂时关闭流式处理（避免 ChatTongyi 库底层 tool_calls 处理 bug）
)

# 2. 在较新的 LangChain 1.x 版本中，AgentExecutor 已被移除或弃用，
# 官方推荐直接使用 create_agent 函数来构建底层基于 LangGraph 的 Agent 实例。
agent = create_agent(
    model=llm,
    tools=tools
)

## 4. 发起调用

In [55]:
print("正在调用大模型...")

# LangGraph 底层的 agent 接受的输入结构通常为包含 messages 的字典
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "成都今天天气如何？现在几点了？"}
    ]
})

print("\n最终回答：")
print(response)
# 返回值包含完整的消息记录，我们取最后一条模型回复作为结果
print(response["messages"][-1].content)

print("\n===========================\n")
print("正在演示流式输出大模型响应...")
print("(注：对于 ChatTongyi，如果你需要在调用 Agent 时也进行流式输出，必须避免它在同一请求中触发多个工具回调（底层 SDK 会产生 IndexError），建议使用独立的普通大模型对象进行流式对话。)")


正在调用大模型...

最终回答：
{'messages': [HumanMessage(content='成都今天天气如何？现在几点了？', additional_kwargs={}, response_metadata={}, id='3d204dfe-795a-4ac0-8c20-38be637745ee'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_76cea0927ab545dd9637a1', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"location": "成都"}'}}, {'index': 1, 'id': 'call_7b86532574614256bd4fcf', 'type': 'function', 'function': {'name': 'get_current_time', 'arguments': '{}'}}]}, response_metadata={'model_name': 'qwen-plus', 'finish_reason': 'tool_calls', 'request_id': '4060c1f5-d1cb-9bc4-8258-2b9376b32ab4', 'token_usage': {'input_tokens': 194, 'output_tokens': 36, 'total_tokens': 230, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--019e724e-09de-76a0-beff-f0774874f797-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '成都'}, 'id': 'call_76cea0927ab545dd9637a1', 'type': 'tool_call'}, {'name': 'get_current_time', 'args': {}, 'id': 'call_7b86532574614256bd4